# Isolation Forest on cybersecurity.csv

This notebook trains an Isolation Forest model for anomaly detection using the dataset at `cybersecurity.csv`.

- Handles missing values
- Encodes categorical columns
- Trains Isolation Forest
- Saves scored output


In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import joblib

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)


In [ ]:
DATA_PATH = r'cybersecurity.csv'
df = pd.read_csv(DATA_PATH)
df.head()


In [ ]:
df.shape


In [ ]:
# Optional: detect a label/anomaly column if present
label_candidates = [
    'label', 'labels', 'anomaly', 'is_anomaly', 'is_anomalous',
    'fraud', 'is_fraud', 'attack', 'is_attack', 'target', 'y'
]
label_col = None
for c in df.columns:
    if c.lower() in label_candidates:
        label_col = c
        break

label_col


In [ ]:
# Separate features and (optional) label
if label_col:
    y = df[label_col]
    X = df.drop(columns=[label_col])
else:
    y = None
    X = df.copy()

# Handle missing values
num_cols = X.select_dtypes(include=[np.number]).columns
cat_cols = X.select_dtypes(exclude=[np.number]).columns

X[num_cols] = X[num_cols].fillna(X[num_cols].median())
X[cat_cols] = X[cat_cols].fillna('missing')

# One-hot encode categorical columns
X_encoded = pd.get_dummies(X, columns=cat_cols, drop_first=False)
X_encoded.shape


In [ ]:
# Train Isolation Forest
iso = IsolationForest(
    n_estimators=200,
    contamination=0.02,
    max_samples='auto',
    random_state=42,
    n_jobs=-1
)

iso.fit(X_encoded)


In [ ]:
# Score data
scores = iso.decision_function(X_encoded)  # higher is more normal
pred = iso.predict(X_encoded)  # 1 = normal, -1 = anomaly

df_scored = df.copy()
df_scored['anomaly_score'] = scores
df_scored['is_anomaly'] = (pred == -1).astype(int)

df_scored.head()


In [ ]:
# If a label is present, show evaluation
if y is not None:
    # Normalize labels to 1 (anomaly) and 0 (normal) if possible
    y_norm = y.copy()
    if y_norm.dtype == object:
        y_norm = y_norm.astype(str).str.lower().map({
            'anomaly': 1, 'anomalous': 1, 'fraud': 1, 'attack': 1, '1': 1, 'true': 1, 'yes': 1,
            'normal': 0, 'benign': 0, '0': 0, 'false': 0, 'no': 0
        })
    y_norm = pd.to_numeric(y_norm, errors='coerce')

    if y_norm.isna().mean() < 0.5:
        print(classification_report(y_norm, df_scored['is_anomaly']))
    else:
        print('Label column found, but could not normalize for evaluation.')
else:
    print('No label column detected; skipping evaluation.')


In [ ]:
# Save results and model
df_scored.to_csv('isolation_forest_results.csv', index=False)
joblib.dump(iso, 'isolation_forest_model.joblib')

'Saved: isolation_forest_results.csv, isolation_forest_model.joblib'
